# Numerical Methods for Ordinary Differential Equations
# Méthodes Numériques pour les EDO

**AIMS Master's Programme — ODE Course**

We implement and compare the major numerical methods for solving initial value problems: Euler's method, Heun's method (improved Euler), and the classical Runge-Kutta 4th order scheme. We study error behaviour, step size effects, and stiff equations (équations raides).

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import time

plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12})

## 1. Euler's Method (Méthode d'Euler)

The simplest one-step method for $y' = f(t, y)$:
$$y_{n+1} = y_n + h\,f(t_n, y_n)$$

- **Local truncation error** (erreur locale de troncature): $O(h^2)$
- **Global error** (erreur globale): $O(h)$ — the method is first-order

The method is derived from the Taylor expansion: $y(t+h) = y(t) + h\,y'(t) + \frac{h^2}{2}y''(\xi)$.

In [ ]:
def euler(f, t_span, y0, h):
    """Forward Euler method."""
    t0, tf = t_span
    t = np.arange(t0, tf + h/2, h)
    y = np.zeros((len(t), len(np.atleast_1d(y0))))
    y[0] = np.atleast_1d(y0)
    for i in range(len(t) - 1):
        y[i+1] = y[i] + h * np.atleast_1d(f(t[i], y[i]))
    return t, y

# Test: dy/dt = -y, y(0)=1, exact: y=e^{-t}
f_decay = lambda t, y: -y
t_exact = np.linspace(0, 5, 500)
y_exact = np.exp(-t_exact)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Solution comparison
ax1.plot(t_exact, y_exact, 'k-', linewidth=2, label='Exact')
for h_val, color in [(1.0, 'red'), (0.5, 'orange'), (0.2, 'green'), (0.05, 'blue')]:
    t_e, y_e = euler(f_decay, (0, 5), [1.0], h_val)
    ax1.plot(t_e, y_e[:, 0], 'o-', color=color, markersize=3, label=f'h={h_val}')

ax1.set_xlabel('$t$'); ax1.set_ylabel('$y(t)$')
ax1.set_title('Euler method: convergence with decreasing $h$')
ax1.legend(); ax1.grid(True, alpha=0.3)

# Error analysis: confirm O(h) convergence
h_values = np.logspace(-4, 0, 40)
errors_euler = []
for h_val in h_values:
    t_e, y_e = euler(f_decay, (0, 5), [1.0], h_val)
    errors_euler.append(abs(y_e[-1, 0] - np.exp(-5)))

ax2.loglog(h_values, errors_euler, 'bo-', markersize=3, label='Euler error at $t=5$')
ax2.loglog(h_values, h_values, 'r--', label='$O(h)$ reference')
ax2.set_xlabel('Step size $h$ (pas)'); ax2.set_ylabel('Global error')
ax2.set_title('Euler: first-order convergence')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
# Figure: Left — Euler solutions converge to exact as h decreases.
# Right — log-log error plot confirms O(h) convergence rate.

## 2. Improved Euler / Heun's Method (Méthode de Heun)

Heun's method is a **predictor-corrector** scheme:
1. **Predict**: $\tilde{y}_{n+1} = y_n + h\,f(t_n, y_n)$
2. **Correct**: $y_{n+1} = y_n + \frac{h}{2}\bigl[f(t_n, y_n) + f(t_{n+1}, \tilde{y}_{n+1})\bigr]$

This is a second-order method: global error is $O(h^2)$. It can be seen as a 2-stage Runge-Kutta method.

In [ ]:
def heun(f, t_span, y0, h):
    """Heun's method (improved Euler / méthode de Heun)."""
    t0, tf = t_span
    t = np.arange(t0, tf + h/2, h)
    y = np.zeros((len(t), len(np.atleast_1d(y0))))
    y[0] = np.atleast_1d(y0)
    for i in range(len(t) - 1):
        k1 = np.atleast_1d(f(t[i], y[i]))
        y_pred = y[i] + h * k1
        k2 = np.atleast_1d(f(t[i] + h, y_pred))
        y[i+1] = y[i] + (h / 2) * (k1 + k2)
    return t, y

# Verify second-order convergence
errors_heun = []
for h_val in h_values:
    t_h, y_h = heun(f_decay, (0, 5), [1.0], h_val)
    errors_heun.append(abs(y_h[-1, 0] - np.exp(-5)))

plt.figure()
plt.loglog(h_values, errors_euler, 'bo-', markersize=3, label='Euler $O(h)$')
plt.loglog(h_values, errors_heun, 'rs-', markersize=3, label='Heun $O(h^2)$')
plt.loglog(h_values, h_values, 'b--', alpha=0.5, label='$h$')
plt.loglog(h_values, h_values**2, 'r--', alpha=0.5, label='$h^2$')
plt.xlabel('Step size $h$'); plt.ylabel('Global error at $t=5$')
plt.title('Convergence: Euler vs Heun')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
# Figure: Heun's method achieves O(h^2) convergence — much better than Euler for the same h.

## 3. Classical Runge-Kutta 4th Order (RK4)

The workhorse of numerical ODE solving. Four function evaluations per step:

$$k_1 = f(t_n, y_n)$$
$$k_2 = f\!\left(t_n + \frac{h}{2},\; y_n + \frac{h}{2}k_1\right)$$
$$k_3 = f\!\left(t_n + \frac{h}{2},\; y_n + \frac{h}{2}k_2\right)$$
$$k_4 = f(t_n + h,\; y_n + h\,k_3)$$
$$y_{n+1} = y_n + \frac{h}{6}(k_1 + 2k_2 + 2k_3 + k_4)$$

Global error: $O(h^4)$. This is remarkably accurate for its computational cost.

In [ ]:
def rk4(f, t_span, y0, h):
    """Classical 4th-order Runge-Kutta method."""
    t0, tf = t_span
    t = np.arange(t0, tf + h/2, h)
    y = np.zeros((len(t), len(np.atleast_1d(y0))))
    y[0] = np.atleast_1d(y0)
    for i in range(len(t) - 1):
        k1 = np.atleast_1d(f(t[i], y[i]))
        k2 = np.atleast_1d(f(t[i] + h/2, y[i] + h/2 * k1))
        k3 = np.atleast_1d(f(t[i] + h/2, y[i] + h/2 * k2))
        k4 = np.atleast_1d(f(t[i] + h, y[i] + h * k3))
        y[i+1] = y[i] + (h/6) * (k1 + 2*k2 + 2*k3 + k4)
    return t, y

# Verify fourth-order convergence
errors_rk4 = []
for h_val in h_values:
    t_r, y_r = rk4(f_decay, (0, 5), [1.0], h_val)
    errors_rk4.append(abs(y_r[-1, 0] - np.exp(-5)))

plt.figure()
plt.loglog(h_values, errors_euler, 'bo-', markersize=3, label='Euler $O(h)$')
plt.loglog(h_values, errors_heun, 'rs-', markersize=3, label='Heun $O(h^2)$')
plt.loglog(h_values, errors_rk4, 'g^-', markersize=3, label='RK4 $O(h^4)$')
plt.loglog(h_values, h_values, 'b--', alpha=0.3)
plt.loglog(h_values, h_values**2, 'r--', alpha=0.3)
plt.loglog(h_values, h_values**4, 'g--', alpha=0.3)
plt.xlabel('Step size $h$'); plt.ylabel('Global error at $t=5$')
plt.title('Convergence comparison: Euler vs Heun vs RK4')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
# Figure: RK4 error decreases as h^4 — dramatically more accurate.

## 4. Head-to-Head Comparison on a Challenging Problem

Let us compare all three methods on the nonlinear ODE $y' = -10(y - \sin t)$, $y(0) = 1$, which has a moderately stiff transient. This makes step-size choice critical for Euler.

In [ ]:
f_challenge = lambda t, y: -10 * (y - np.sin(t))

# Reference solution with very small step
t_ref, y_ref = rk4(f_challenge, (0, 5), [1.0], 0.0001)

h_test = 0.15
t_eu, y_eu = euler(f_challenge, (0, 5), [1.0], h_test)
t_he, y_he = heun(f_challenge, (0, 5), [1.0], h_test)
t_rk, y_rk = rk4(f_challenge, (0, 5), [1.0], h_test)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))

ax1.plot(t_ref, y_ref[:, 0], 'k-', linewidth=2, label='Reference')
ax1.plot(t_eu, y_eu[:, 0], 'bo-', markersize=4, label=f'Euler (h={h_test})', alpha=0.7)
ax1.plot(t_he, y_he[:, 0], 'rs-', markersize=4, label=f'Heun (h={h_test})', alpha=0.7)
ax1.plot(t_rk, y_rk[:, 0], 'g^-', markersize=4, label=f'RK4 (h={h_test})', alpha=0.7)
ax1.set_xlabel('$t$'); ax1.set_ylabel('$y(t)$')
ax1.set_title("$y' = -10(y - \\sin t)$: method comparison")
ax1.legend(); ax1.grid(True, alpha=0.3)

# Error vs number of function evaluations (cost-accuracy tradeoff)
h_range = np.logspace(-3, -0.5, 25)
methods = [
    ('Euler', euler, 1, 'bo-'),
    ('Heun', heun, 2, 'rs-'),
    ('RK4', rk4, 4, 'g^-'),
]

y_true_at_5 = y_ref[-1, 0]
for name, method, n_evals_per_step, style in methods:
    errs, costs = [], []
    for h_val in h_range:
        t_m, y_m = method(f_challenge, (0, 5), [1.0], h_val)
        errs.append(abs(y_m[-1, 0] - y_true_at_5))
        costs.append(len(t_m) * n_evals_per_step)
    ax2.loglog(costs, errs, style, markersize=4, label=name)

ax2.set_xlabel('Number of $f$ evaluations')
ax2.set_ylabel('Error at $t=5$')
ax2.set_title('Cost-accuracy tradeoff (compromis coût-précision)')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
# Figure: Top — solution comparison. Bottom — RK4 gives far better accuracy
# per function evaluation than Euler or Heun.

## 5. `scipy.integrate.solve_ivp` — Adaptive Methods

In practice, we use **adaptive** methods that automatically adjust the step size to maintain a target error tolerance. `solve_ivp` offers:

| Method | Order | Use case |
|:---|:---:|:---|
| `RK45` (default) | 4(5) | General purpose, non-stiff |
| `RK23` | 2(3) | Low accuracy, fast |
| `Radau` | 5 | Stiff problems (implicit) |
| `BDF` | 1-5 | Stiff problems (multistep) |
| `LSODA` | — | Auto-detects stiffness |

In [ ]:
# Compare solve_ivp methods on a non-stiff problem
f_test2 = lambda t, y: [-np.sin(t) * y[0] + np.cos(t)]
t_span = (0, 20)
y0_test = [1.0]

methods_ivp = ['RK45', 'RK23', 'Radau', 'BDF', 'LSODA']
colors = ['blue', 'orange', 'red', 'green', 'purple']

plt.figure(figsize=(10, 6))
for method, color in zip(methods_ivp, colors):
    t0 = time.time()
    sol = solve_ivp(f_test2, t_span, y0_test, method=method,
                    t_eval=np.linspace(*t_span, 500))
    elapsed = time.time() - t0
    plt.plot(sol.t, sol.y[0], color=color, linewidth=1.5,
             label=f'{method} ({elapsed*1000:.1f} ms)')

plt.xlabel('$t$'); plt.ylabel('$y(t)$')
plt.title('solve_ivp: different methods on a non-stiff problem')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
# Figure: All methods agree — for non-stiff problems the choice matters less.

## 6. Stiff Equations (Équations raides) and Method Choice

A system is **stiff** when it contains dynamics at very different time scales. The classic example:

$$y' = -1000(y - \sin t) + \cos t, \quad y(0) = 0$$

The solution has a fast transient $\sim e^{-1000t}$ and a slow oscillation $\sim \sin t$. Explicit methods (like RK45) must take tiny steps to remain stable, while implicit methods (Radau, BDF) handle this efficiently.

**The stiffness ratio** (rapport de raideur) is the ratio of the largest to smallest eigenvalue magnitudes of the Jacobian.

In [ ]:
# Stiff equation: y' = -1000(y - sin(t)) + cos(t)
f_stiff = lambda t, y: [-1000*(y[0] - np.sin(t)) + np.cos(t)]
jac_stiff = lambda t, y: [[-1000]]  # Jacobian for implicit methods

t_span_s = (0, 5)
t_eval_s = np.linspace(*t_span_s, 1000)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Explicit RK45 — needs many steps
sol_rk45 = solve_ivp(f_stiff, t_span_s, [0.0], method='RK45', t_eval=t_eval_s)
# Implicit Radau — handles stiffness efficiently
sol_radau = solve_ivp(f_stiff, t_span_s, [0.0], method='Radau',
                      jac=jac_stiff, t_eval=t_eval_s)
# BDF
sol_bdf = solve_ivp(f_stiff, t_span_s, [0.0], method='BDF',
                    jac=jac_stiff, t_eval=t_eval_s)

ax1.plot(sol_rk45.t, sol_rk45.y[0], 'b-', label='RK45', linewidth=1.5)
ax1.plot(sol_radau.t, sol_radau.y[0], 'r--', label='Radau', linewidth=1.5)
ax1.plot(sol_bdf.t, sol_bdf.y[0], 'g:', label='BDF', linewidth=2)
ax1.plot(t_eval_s, np.sin(t_eval_s), 'k--', alpha=0.3, label='$\\sin(t)$ (attractor)')
ax1.set_xlabel('$t$'); ax1.set_ylabel('$y(t)$')
ax1.set_title('Stiff ODE: all methods agree on solution')
ax1.legend(); ax1.grid(True, alpha=0.3)

# Show that explicit Euler is UNSTABLE for large h on stiff problems
h_values_stiff = [0.003, 0.002, 0.001, 0.0005]
for h_val in h_values_stiff:
    t_eu, y_eu = euler(f_stiff, (0, 0.05), [0.0], h_val)
    ax2.plot(t_eu, y_eu[:, 0], 'o-', markersize=2, label=f'Euler h={h_val}')

ax2.set_xlabel('$t$'); ax2.set_ylabel('$y(t)$')
ax2.set_title('Euler on stiff ODE: stability requires very small $h$')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print step counts
print(f"Steps taken — RK45: {sol_rk45.t.shape[0]}, Radau: {sol_radau.t.shape[0]}, BDF: {sol_bdf.t.shape[0]}")
# Figure: Left — all methods find the same solution. Right — Euler may become
# unstable unless h < 2/1000 = 0.002 (stability boundary).

## 7. Exercise: Stiff Chemical Kinetics (Cinétique chimique raide)

The Robertson chemical kinetics problem is a classic stiff test:

$$\frac{dy_1}{dt} = -0.04\,y_1 + 10^4\,y_2\,y_3$$
$$\frac{dy_2}{dt} = 0.04\,y_1 - 10^4\,y_2\,y_3 - 3\times10^7\,y_2^2$$
$$\frac{dy_3}{dt} = 3\times10^7\,y_2^2$$

with $y_1(0) = 1$, $y_2(0) = 0$, $y_3(0) = 0$. The stiffness ratio is approximately $10^{11}$.

**Tasks:**
1. Solve with `Radau` and `BDF` for $t \in [0, 10^{11}]$ (use logarithmic time).
2. Try `RK45` — observe that it either fails or takes enormous time.
3. Provide the Jacobian to speed up implicit methods.
4. Plot $y_1(t)$ and $10^4 y_2(t)$ on a semilog-$x$ scale.
5. Verify conservation: $y_1 + y_2 + y_3 = 1$ for all $t$.

In [ ]:
# Robertson problem — starter code
def robertson(t, y):
    y1, y2, y3 = y
    dy1 = -0.04 * y1 + 1e4 * y2 * y3
    dy2 = 0.04 * y1 - 1e4 * y2 * y3 - 3e7 * y2**2
    dy3 = 3e7 * y2**2
    return [dy1, dy2, dy3]

def robertson_jac(t, y):
    y1, y2, y3 = y
    return [
        [-0.04, 1e4 * y3, 1e4 * y2],
        [0.04, -1e4 * y3 - 6e7 * y2, -1e4 * y2],
        [0, 6e7 * y2, 0]
    ]

y0_rob = [1.0, 0.0, 0.0]
t_span_rob = (1e-5, 1e11)
t_eval_rob = np.logspace(-5, 11, 1000)

sol_rob = solve_ivp(robertson, t_span_rob, y0_rob, method='Radau',
                    jac=robertson_jac, t_eval=t_eval_rob, rtol=1e-8, atol=1e-10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.semilogx(sol_rob.t, sol_rob.y[0], 'b-', label='$y_1$', linewidth=2)
ax1.semilogx(sol_rob.t, 1e4 * sol_rob.y[1], 'r-', label='$10^4 \\cdot y_2$', linewidth=2)
ax1.semilogx(sol_rob.t, sol_rob.y[2], 'g-', label='$y_3$', linewidth=2)
ax1.set_xlabel('Time $t$ (log scale)'); ax1.set_ylabel('Concentration')
ax1.set_title('Robertson chemical kinetics (Radau)')
ax1.legend(); ax1.grid(True, alpha=0.3)

# Conservation check
total_rob = sol_rob.y[0] + sol_rob.y[1] + sol_rob.y[2]
ax2.semilogx(sol_rob.t, total_rob - 1, 'k-')
ax2.set_xlabel('Time $t$ (log scale)')
ax2.set_ylabel('$y_1 + y_2 + y_3 - 1$')
ax2.set_title('Conservation check')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
# Figure: Robertson problem spans 16 orders of magnitude in time —
# only implicit methods like Radau can handle this efficiently.